# 01 — Setup & data shards (Colab T4)

Clone **midi-llm** + **musicinstruct**, install dependencies, authenticate Hugging Face, and build **Stage S3 edit shards** for all four representation arms.

**Runtime:** GPU → T4 (or better).

In [ ]:
# @title Configuration — edit if your forks differ
MIDI_LLM_REPO = "https://github.com/juliagsy/midi-llm.git"
MUSICINSTRUCT_REPO = "https://github.com/juliagsy/musicinstruct.git"
BRANCH = "main"  # or your feature branch

WORK = "/content"
MIDI_LLM = f"{WORK}/midi-llm"
MUSICINSTRUCT = f"{WORK}/musicinstruct"
RUN_ID = "pilot_v1"  # reuse in notebooks 02 and 03
RUNS_ROOT = f"{WORK}/runs/{RUN_ID}"
SHARDS_ROOT = f"{WORK}/data/shards/{RUN_ID}"

In [ ]:
# @title Hugging Face token (Llama 3.2 license required)
import os
from getpass import getpass

if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass("HF token (hf_...): ")

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

In [ ]:
# @title Clone repos
!rm -rf {MIDI_LLM} {MUSICINSTRUCT}
!git clone --branch {BRANCH} --depth 1 {MIDI_LLM_REPO} {MIDI_LLM}
!git clone --branch {BRANCH} --depth 1 {MUSICINSTRUCT_REPO} {MUSICINSTRUCT}
%cd {MIDI_LLM}

In [ ]:
# @title Install midi-llm + musicinstruct (T4-friendly stack)
!pip install -q -e ".[repr,data,train]"
!pip install -q -e "{MUSICINSTRUCT}"
# Optional AMT arm (GitHub install)
!pip install -q "git+https://github.com/jthickstun/anticipation.git" mido

In [ ]:
# @title GPU check
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    raise RuntimeError("Enable GPU: Runtime → Change runtime type → T4 GPU")

In [ ]:
# @title Generate MIDI-Instruct pilot data (if not in repo)
import os
from pathlib import Path

manifest = Path(MUSICINSTRUCT) / "data/pilot/pilot.jsonl"
if not manifest.is_file():
    !cd {MUSICINSTRUCT} && musicinstruct generate-pilot --output-dir data/pilot --target 300
    !cd {MUSICINSTRUCT} && musicinstruct validate data/pilot/pilot.jsonl
    !cd {MUSICINSTRUCT} && musicinstruct self-test data/pilot/pilot.jsonl
else:
    print("Using existing manifest:", manifest)

In [ ]:
# @title Build S3 edit shards (train + validation) for all representations
import os
from pathlib import Path

os.makedirs(SHARDS_ROOT, exist_ok=True)
manifest = f"{MUSICINSTRUCT}/data/pilot/pilot.jsonl"
reprs = ["abc", "remi", "octuple", "amt"]

for split in ("train", "validation"):
    for repr_name in reprs:
        out = f"{SHARDS_ROOT}/edit_{repr_name}_{split}.jsonl"
        !midi-llm build-instruct-shard {manifest} --repr {repr_name} --output {out} --split {split}

print("Shards written to:", SHARDS_ROOT)
!ls -la {SHARDS_ROOT}

In [ ]:
# @title Optional — GigaMIDI syntax shard (S0, no Lakh)
BUILD_GIGAMIDI = False  # @param {type:"boolean"}
GIGAMIDI_LIMIT = 200  # @param {type:"integer"}

if BUILD_GIGAMIDI:
    out = f"{SHARDS_ROOT}/syntax_remi_gigamidi.jsonl"
    !midi-llm build-gigamidi-shard --repr remi --output {out} --limit {GIGAMIDI_LIMIT}
else:
    print("Skipped GigaMIDI (set BUILD_GIGAMIDI=True to enable)")

In [ ]:
# @title Training time estimate on this GPU
import json
from midi_llm.train.estimate import estimate_training

for steps in (50, 300, 3000):
    report = estimate_training(repr_name="remi", steps=steps, seq_len=1024, run_benchmark=False)
    est = report["estimate"]
    print(f"steps={steps}: ~{est['estimated_hours']} h ({est['seconds_per_step']} s/step, {est['notes']})")

## Next

Open **`02_train_lora_sft.ipynb`** with the same `RUN_ID` (`pilot_v1`).